Environment & Dependency Setup

In [11]:
# Cell 1: Imports and folder creation
import os
import pandas as pd
from google_play_scraper import Sort, reviews_all

os.makedirs('../data/raw', exist_ok=True)
print("✅ Project folders verified. Upstream configurations loaded.")

✅ Project folders verified. Upstream configurations loaded.


Define App Targets

In [16]:
# Cell 2: Define application target metadata
BANKS_CONFIG = {
    "CBE": {
        "app_id": "prod.cbe.birr", 
        "name": "Commercial Bank of Ethiopia"
    },
    "BOA": {
        "app_id": "com.boa.boaMobileBanking", 
        "name": "Bank of Abyssinia"
    },
    "Dashen": {
        "app_id": "com.dashen.dashensuperapp", 
        "name": "Dashen Bank"
    }
}
print("📋 Target banks registered successfully.")

📋 Target banks registered successfully.


Scrape the Raw Reviews

In [17]:
# Cell 3: Raw extraction sequence
all_scraped_dfs = []

for key, config in BANKS_CONFIG.items():
    print(f"🔄 Fetching raw reviews for {config['name']}...")
    try:
        # Fetching raw reviews
        raw_list = reviews_all(
            config['app_id'],
            lang='en',       # Target English reviews
            country='et',    # Filter specifically for the Ethiopian Play Store region
            sort=Sort.NEWEST
        )
        
        # Turn list into a standard DataFrame
        df_temp = pd.DataFrame(raw_list)
        
        # Map raw keys into the clean 5 columns needed for the assignment
        df_mapped = pd.DataFrame({
            'review': df_temp['content'],
            'rating': df_temp['score'],
            'date': df_temp['at'],
            'bank': config['name'],
            'source': 'Google Play'
        })
        
        # Take a safe slice of 450 reviews per bank (leaves room for duplicate cleaning)
        all_scraped_dfs.append(df_mapped.head(450))
        print(f"✅ Gathered {len(df_mapped.head(450))} reviews for {key}.")
        
    except Exception as e:
        print(f"❌ Network or configuration error on {key}: {str(e)}")

# Combine everything into one single dataframe
raw_combined_df = pd.concat(all_scraped_dfs, ignore_index=True)
print(f"\n📦 Initial harvesting complete! Total raw rows collected: {len(raw_combined_df)}")

🔄 Fetching raw reviews for Commercial Bank of Ethiopia...
✅ Gathered 450 reviews for CBE.
🔄 Fetching raw reviews for Bank of Abyssinia...
✅ Gathered 450 reviews for BOA.
🔄 Fetching raw reviews for Dashen Bank...
✅ Gathered 450 reviews for Dashen.

📦 Initial harvesting complete! Total raw rows collected: 1350


Clean and Preprocess the Data

In [18]:
# Cell 4: Preprocessing and Data Quality Engine
print(f"🧼 Initial row count: {len(raw_combined_df)}")



🧼 Initial row count: 1350


In [19]:
# 1. Handle Missing Values: Drop rows with missing text or rating
cleaned_df = raw_combined_df.dropna(subset=['review', 'rating']).copy()
print(f"• Rows after removing blank text: {len(cleaned_df)}")


• Rows after removing blank text: 1350


In [20]:
# 2. Deduplicate: Remove repeating or overlapping review entries
cleaned_df.drop_duplicates(subset=['review', 'bank'], inplace=True)
print(f"• Rows after dropping duplicates: {len(cleaned_df)}")


• Rows after dropping duplicates: 1093


In [21]:
# 3. Normalize Date Format: Change timestamps to clean YYYY-MM-DD strings
cleaned_df['date'] = pd.to_datetime(cleaned_df['date']).dt.strftime('%Y-%m-%d')


In [22]:
# 4. Save to CSV inside data/raw/ directory
output_path = '../data/raw/cleaned_reviews.csv'
cleaned_df.to_csv(output_path, index=False)
print(f"\n💾 Cleaned dataset successfully written to: {output_path}")


💾 Cleaned dataset successfully written to: ../data/raw/cleaned_reviews.csv


Instant KPI Audit Display

In [23]:
# Cell 5: Inspection Grid
print("📊 --- COLLECTION SUMMARY ---")
display(cleaned_df.groupby('bank').size().reset_index(name='Cleaned Reviews Count'))

print("\n👀 --- FIRST 3 ROWS PREVIEW ---")
display(cleaned_df.head(3))

📊 --- COLLECTION SUMMARY ---


,bank,Cleaned Reviews Count
0,Bank of Abyssinia,370
1,Commercial Bank of Ethiopia,344
2,Dashen Bank,379



👀 --- FIRST 3 ROWS PREVIEW ---


,review,rating,date,bank,source
0,CBE One of the best in Ethiopia bink,4,2026-05-15,Commercial Bank of Ethiopia,Google Play
1,good 👍,5,2026-05-13,Commercial Bank of Ethiopia,Google Play
2,fast & safe banking changes your carrier .!,5,2026-05-10,Commercial Bank of Ethiopia,Google Play
